# Native quantum autoencoder

Train a four-qubit encoder with two coded and two trash qubits. Keep the source's ten pretrained weights, three domain-wall examples, 4096-shot evaluations, batch size two, L1 loss and SGD learning rate 0.3. Each of the two independent training routes executes **one epoch**, including a final single-example batch.

Native Python qfunc expresses encoding, composition, swap testing, RESET and adjoint. No Qmod source, importer, remote service or credentials are required. See [WORKFLOW.md](WORKFLOW.md) for the exact source mapping.

## Inspect encoding independently of the training driver

The editable encoder below defines data rotations, both entangling layers and
the final trash rotations. It synthesizes one checkpoint binding and reads its
named four-bit output before the full training routes run. The compiler supplies
register mapping and gate construction; the user supplies the ansatz and loss.
The linked workflow composes that encoder with swap testing or RESET/adjoint.

Try another one of the three declared domain-wall inputs in the preview. Their
encoder distributions can differ even when training minimizes a common trash
cost. Changing the ansatz requires updating the independent gradient and inverse
checks. RESET is a channel and its subsequent decoder can be mixed; a statevector
from the encoder preview must not be used as a fabricated pure decoded result.


In [ ]:
import math
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import CX, Output, QArray, QBit, R, allocate, qfunc


@qfunc
def ansatz(weights: list[float], q: QArray[QBit]) -> None:
    for layer in range(2):
        for i in range(4):
            R(weights[4 * layer + i], math.pi / 2, q[i])
        for i in range(3):
            CX(q[i], q[i + 1])
    for i in range(2):
        R(weights[8 + i], math.pi / 2, q[2 + i])


@qfunc
def encode(data: list[float], weights: list[float], q: QArray[QBit]) -> None:
    for i in range(4):
        R(math.pi * data[i], math.pi / 2, q[i])
    ansatz(weights, q)


@qfunc
def encode_input(
    data: list[float], weights: list[float], q: Output[QArray[QBit]]
) -> None:
    allocate(4, q)
    encode(data, weights, q)


import torch
from matplotlib import pyplot as plt

from examples.synthesis.algorithms.machine_learning.quantum_autoencoder.overview.workflow import (
    DOMAIN_WALL,
    SHOTS,
    build_program,
    initial_weights,
    plot_workflow,
    run_workflow,
)

print("Training inputs:", DOMAIN_WALL)
print("Checkpoint:", initial_weights().tolist())
print("Shots per execution:", SHOTS)

preview = synth.synthesize(
    encode_input,
    classical_inputs={
        "data": list(DOMAIN_WALL[0]),
        "weights": initial_weights().tolist(),
    },
    max_qubits=4,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
print(
    "Checkpoint encoder probabilities:",
    dict(preview.probabilities(fq.run(preview), registers=("q",))),
)


## Two independent training routes

The swap route measures P(test=1). The alternative measures the expected number of nonzero trash bits. Both restart from the same checkpoint. All ten weight derivatives use actual quantum parameter-shift executions with 4096 sampled outcomes; static float synthesis is not a generic autograd QLayer. Explicit local seeds and float64 arithmetic replace the source's unspecified sampling/loader seeds and float32 arithmetic; identical source sampled trajectories are not claimed.

In [ ]:
workflow = run_workflow()
for training in (workflow.swap, workflow.trash):
    print(training.cost, "batch sizes:", [len(step.data) for step in training.steps])
    print("Sampled L1 losses:", [step.loss for step in training.steps])
    print("Trained weights:", training.weights.tolist())
print("Quantum executions:", workflow.quantum_evaluations)


## RESET decoding and anomaly detection

RESET discards the two trash values through a physical quantum channel; applying the inverse encoder afterward generally produces a mixed density matrix. Read modal logical bits, not a fabricated pure state. The swap reference is retained entangled environment after drop, not clean work. Detect all five source inputs with the original strict 0.01 threshold.

In [ ]:
for decoded in workflow.decoded:
    print("Input/decoded:", decoded.data, decoded.modal_bits)
    density = decoded.density.reshape(16, 16)
    print("Decoded purity:", float(torch.trace(density @ density).real))
for anomaly in workflow.anomalies:
    print("Input/cost/anomaly:", anomaly.data, anomaly.cost, anomaly.detected)


## Measured results

Plots show the actual two minibatch losses per route and five sampled anomaly costs. No monotonic convergence or quantum advantage is claimed.

In [ ]:
plt.show()
figure = plot_workflow(workflow)
figure


## Resource boundaries

Report resources per bound circuit separately from the complete 134-execution workload. Swap testing uses seven physical bits (four data, one test, two environment); the encoder and decoder use four. All paths use zero clean work bits. This is a local CPU `single_device_fast_path`, not a distributed scalability claim.

In [ ]:
for mode in ("swap", "trash", "decode"):
    program = build_program(
        torch.tensor(DOMAIN_WALL[0], dtype=torch.float64),
        workflow.swap.weights,
        mode=mode,
    )
    print(mode, program.execution.state_mode, program.report["resources"])
    print("Clean work bits:", program.report["workspace_qubits"])
